# Hybrid GraphRAG Pipeline - Full SOTA Implementation

This notebook demonstrates the full hybrid pipeline combining:
- BM25 keyword retrieval
- Dense retrieval (BGE-M3 + FAISS)
- Citation graph retrieval (PPR ranking)
- Weighted Reciprocal Rank Fusion (RRF)
- Cross-encoder reranking (BGE-reranker-v2-m3)
- LLM verification (Qwen2.5-7B GGUF)

## Pipeline Components
1. **Retrieval Stage**: BM25 + Dense + Graph → multiple signal sources
2. **Fusion Stage**: Combine signals using Weighted RRF
3. **Reranking Stage**: Re-rank fused results with cross-encoder
4. **Verification Stage**: Filter results with LLM verifier
5. **Normalization**: Canonicalize all citations

## Experiment Presets
- `exp_baseline`: BM25 only
- `exp_dense_only`: Dense retrieval only
- `exp_bm25_dense`: BM25 + Dense fusion
- `exp_full_retrieval`: BM25 + Dense + Graph
- `exp_full_rrf`: Full retrieval + RRF fusion
- `exp_full_reranker`: Full retrieval + RRF + Reranker
- `exp_full_pipeline`: Full pipeline with all components

## 1. Setup & Configuration

In [ ]:
import os
import sys
from pathlib import Path

# Configuration
DATASET_MODE = "val"  # Change to "test" for final submission
DATA_PATH = Path("data")
OUTPUT_PATH = Path("output")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Load presets
from src.omnilex.retrieval.ablation.config import ExperimentConfig, EXPERIMENT_PRESETS
from src.omnilex.retrieval.ablation.runner import ExperimentRunner
from src.omnilex.retrieval.ablation.reporter import ResultsReporter

print(f"Available presets: {list(EXPERIMENT_PRESETS.keys())}")

## 2. Load Queries

In [ ]:
import pandas as pd

QUERY_FILE = DATA_PATH / f"{DATASET_MODE}.csv"
if not QUERY_FILE.exists():
    raise FileNotFoundError(f"Query file not found: {QUERY_FILE}")

test_df = pd.read_csv(QUERY_FILE)
print(f"Loaded {len(test_df)} queries from {QUERY_FILE}")

# Format queries for runner
formatted_queries = [
    {"id": row["query_id"], "query": row["query"]} for _, row in test_df.iterrows()
]

# Load ground truth if available
ground_truth = None
if "gold_citations" in test_df.columns:
    ground_truth = {}
    for _, row in test_df.iterrows():
        citations = [c.strip() for c in str(row["gold_citations"]).split(";") if c.strip()]
        ground_truth[row["query_id"]] = citations
    print(f"Loaded ground truth for {len(ground_truth)} queries")

test_df.head()

## 3. Run Full Pipeline Experiment

In [ ]:
# Run the full pipeline experiment
config = ExperimentConfig.from_preset("exp_full_pipeline")
config.name = "full_pipeline_experiment"

runner = ExperimentRunner(config=config, output_dir=OUTPUT_PATH / "experiments")

print(f"Running experiment: {config.name}")
print(f"Components enabled: {config.components}")

results = runner.run(queries=formatted_queries, ground_truth=ground_truth)

print(f"\nExperiment complete!")
print(f"Aggregate metrics: {results['metrics']}")

## 4. Compare Different Presets

In [ ]:
# Compare multiple presets
presets_to_compare = ["exp_baseline", "exp_bm25_dense", "exp_full_retrieval", "exp_full_pipeline"]

comparison_results = []

for preset_name in presets_to_compare:
    print(f"\n{'=' * 50}")
    print(f"Running preset: {preset_name}")

    config = ExperimentConfig.from_preset(preset_name)
    config.name = preset_name

    runner = ExperimentRunner(config=config, output_dir=OUTPUT_PATH / "experiments" / preset_name)

    results = runner.run(formatted_queries, ground_truth)
    metrics = results["metrics"]

    comparison_results.append(
        {
            "preset": preset_name,
            "macro_f1": metrics.get("macro_f1", 0),
            "macro_precision": metrics.get("macro_precision", 0),
            "macro_recall": metrics.get("macro_recall", 0),
        }
    )

    print(f"Macro F1: {metrics.get('macro_f1', 0):.4f}")

# Show comparison table
comparison_df = pd.DataFrame(comparison_results)
print(f"\n{'=' * 50}")
print("PRESET COMPARISON")
print(f"{'=' * 50}")
print(comparison_df.to_string(index=False))

## 5. Generate Submission

In [ ]:
# Generate final submission from best experiment
submission_path = OUTPUT_PATH / "submission.csv"

# Load the best results (from full pipeline)
import json

best_results_path = (
    OUTPUT_PATH
    / "experiments"
    / "full_pipeline_experiment"
    / "full_pipeline_experiment_results.json"
)

if best_results_path.exists():
    with open(best_results_path, "r") as f:
        best_results = json.load(f)

    # Convert to submission format
    submission_df = pd.DataFrame(best_results["results"])
    submission_df["predicted_citations"] = submission_df["citations"].apply(lambda x: ";".join(x))
    submission_df[["query_id", "predicted_citations"]].to_csv(submission_path, index=False)

    print(f"Submission saved to: {submission_path}")
    print(f"Total queries: {len(submission_df)}")
    print(f"\nSample submission:")
    print(submission_df[["query_id", "predicted_citations"]].head())
else:
    print(f"Best results file not found: {best_results_path}")

## 6. Evaluate Results

In [ ]:
# Run validation if ground truth exists
if ground_truth:
    print("Running evaluation...")
    import subprocess

    result = subprocess.run(
        ["python", "scripts/evaluate_submission.py", str(submission_path)],
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    if result.stderr:
        print(f"Errors: {result.stderr}")
else:
    print("No ground truth available, skipping evaluation")